In [ ]:
# ============================================================
# Task 02 — 乳腺超声影像组学良恶性分类 (BUSI Dataset)  v2
# ============================================================
# 优化项：类别不平衡处理 | 特征相关性分析 | 5折交叉验证 |
#         PR曲线 | Bootstrap AUC CI | 学习曲线 | 统一可视化
# ============================================================

import os
import glob
import warnings

warnings.filterwarnings("ignore")

# ── Numerical ──
import numpy as np
import pandas as pd

# ── Image Processing ──
import cv2
from skimage.measure import regionprops, label
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy, skew, kurtosis

# ── sklearn: preprocessing & feature selection ──
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# ── sklearn: models ──
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# ── sklearn: model selection ──
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, learning_curve, GridSearchCV,
)

# ── sklearn: metrics ──
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
)

# ── Visualization ──
import matplotlib.pyplot as plt
import seaborn as sns

# ── Global Config ──
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})
sns.set_palette("Set2")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 统一配色：三个模型在所有图中颜色一致
MODEL_COLORS = {
    "Logistic Regression": "#2196F3",
    "Random Forest":       "#4CAF50",
    "SVM (RBF)":           "#FF9800",
}

print("Libraries imported successfully! (v2)")


# 乳腺超声影像组学良恶性分类实验

## 实验流程总览

| 步骤 | 内容 | 优化点 |
|------|------|--------|
| **1. 数据加载** | BUSI 数据集（benign / malignant），排除 normal | — |
| **2. 特征提取** | 一阶(17) + 形状(10) + GLCM(18) = **45 维** | — |
| **3. 特征分析** | 相关性热力图 → 冗余检测 | ✅ **新增** |
| **4. 预处理** | 分层划分 → 标准化 → SelectKBest(20) | — |
| **5. 模型训练** | LR + RF + SVM，**class_weight='balanced'** | ✅ **改进** |
| **6. 交叉验证** | 5 折 StratifiedKFold → CV-AUC | ✅ **新增** |
| **7. 评估** | AUC(95%CI) / Acc / Sen / Spe + ROC + **PR曲线** | ✅ **改进** |
| **8. 混淆矩阵** | 三模型对比 | — |
| **9. 特征重要性** | LR 系数 + RF Gini Top-10 | — |
| **10. 学习曲线** | 诊断过拟合 / 欠拟合 | ✅ **新增** |
| **11. SHAP** | 全局 + 个体可解释性 | — |
| **12. 超参数优化** | GridSearchCV 5折 | — |

> **数据路径**：`D:/Research/data/Dataset_BUSI/Dataset_BUSI_with_GT`
>
> **分类任务**：二分类 — benign (0) vs malignant (1)，排除 normal 类
>
> **类别不平衡**：benign 437 vs malignant 210 (≈ 2:1)，使用 `class_weight='balanced'` 处理
>
> **参考文献**：Romeo V et al. *European Radiology*, 2021.

In [ ]:
# ====================
# 1. 配置与数据加载
# ====================

DATA_ROOT = r"D:/Research/data/Dataset_BUSI/Dataset_BUSI_with_GT"
TASK_DIR  = r"D:/Research/medical-ml-learning/Task/Task 02"

def load_busi_dataset(data_root):
    """加载 BUSI 数据集（仅 benign 和 malignant 两类）"""
    records = []
    for label_name, label in [("benign", 0), ("malignant", 1)]:
        folder = os.path.join(data_root, label_name)
        all_files = sorted(glob.glob(os.path.join(folder, "*.png")))
        image_files = [f for f in all_files if "_mask" not in os.path.basename(f)]

        for img_path in image_files:
            base = img_path.rsplit(".", 1)[0]
            mask_paths = sorted(glob.glob(base + "_mask*.png"))
            if not mask_paths:
                continue
            records.append({
                "image_path": img_path,
                "mask_paths": mask_paths,
                "label": label,
                "label_name": label_name,
            })

    df = pd.DataFrame(records)
    print(f"数据集加载完成: 共 {len(df)} 个样本")
    print(df["label_name"].value_counts())
    print(f"\n多 mask 样本数: {df['mask_paths'].apply(len).gt(1).sum()}")
    print(f"类别比例 (benign:malignant) = {df['label'].value_counts()[0]}:{df['label'].value_counts()[1]}")
    return df

df_data = load_busi_dataset(DATA_ROOT)
df_data.head()

In [ ]:
# ====================
# 2. 数据可视化（每类展示 3 个样例：原图 + mask 叠加）
# ====================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, (label_name, label_val) in enumerate([("benign", 0), ("malignant", 1)]):
    subset = df_data[df_data["label"] == label_val].sample(3, random_state=RANDOM_STATE)
    for col, (_, row_data) in enumerate(subset.iterrows()):
        img = cv2.imread(row_data["image_path"], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(row_data["mask_paths"][0], cv2.IMREAD_GRAYSCALE)

        # 原图
        ax = axes[row, col]
        ax.imshow(img, cmap="gray")
        # mask 半透明叠加（红色边界）
        overlay = np.zeros((*img.shape, 3), dtype=np.uint8)
        overlay[mask > 0] = [255, 50, 50]
        ax.imshow(overlay, alpha=0.25)
        ax.set_title(f"{label_name} (#{col+1})", fontsize=12, fontweight="bold")
        ax.axis("off")

plt.suptitle("BUSI Dataset Samples (Image + ROI Overlay)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. 影像组学特征提取

影像组学 (Radiomics) 的核心思想：从医学图像的 ROI（感兴趣区）中提取**高通量定量特征**，
再用机器学习方法挖掘特征与诊断标签之间的关联。

本实验提取三大类共 **45 维**特征：

### 3.1 一阶统计特征 (First-Order Statistics, 17 维)
仅依赖 ROI 内像素强度的分布，不关心空间位置：
- 均值、中位数、标准差、方差
- 偏度 (Skewness)、峰度 (Kurtosis)
- 能量 (Energy)、熵 (Entropy)
- 最小值、最大值、极差
- 百分位数 (P10, P25, P75, P90)、四分位距 (IQR)
- 平均绝对偏差 (MAD)

### 3.2 形状特征 (Shape Features, 10 维)
描述 ROI 的二维几何形态：
- 面积、周长、凸包面积
- 离心率 (Eccentricity)、长宽比 (Aspect Ratio)
- 长轴长度、短轴长度
- 实性度 (Solidity)、延展度 (Extent)、等效直径

### 3.3 GLCM 纹理特征 (18 维)
灰度共生矩阵 (Gray-Level Co-occurrence Matrix) 捕捉像素间的空间关系：
- 对比度 (Contrast)、相异性 (Dissimilarity)
- 同质性 (Homogeneity)、能量 (Energy)
- 相关性 (Correlation)、角二阶矩 (ASM)
- 在距离 d=1,2,3 上分别取 4 个方向的均值 → 6 × 3 = 18 维

In [ ]:
# ====================
# 3. 影像组学特征提取函数
# ====================

def extract_first_order_features(pixels):
    """一阶统计特征（17 维）"""
    f = {}
    f["fo_mean"]     = float(np.mean(pixels))
    f["fo_median"]   = float(np.median(pixels))
    f["fo_std"]      = float(np.std(pixels))
    f["fo_variance"] = float(np.var(pixels))
    f["fo_skewness"] = float(skew(pixels))
    f["fo_kurtosis"] = float(kurtosis(pixels))
    f["fo_energy"]   = float(np.sum(pixels.astype(np.float64) ** 2))
    hist, _ = np.histogram(pixels, bins=256, range=(0, 256))
    prob = hist / hist.sum()
    prob = prob[prob > 0]
    f["fo_entropy"]  = float(-np.sum(prob * np.log2(prob)))
    f["fo_min"]      = float(np.min(pixels))
    f["fo_max"]      = float(np.max(pixels))
    f["fo_range"]    = f["fo_max"] - f["fo_min"]
    f["fo_p10"]      = float(np.percentile(pixels, 10))
    f["fo_p25"]      = float(np.percentile(pixels, 25))
    f["fo_p75"]      = float(np.percentile(pixels, 75))
    f["fo_p90"]      = float(np.percentile(pixels, 90))
    f["fo_iqr"]      = f["fo_p75"] - f["fo_p25"]
    f["fo_mad"]      = float(np.mean(np.abs(pixels - np.mean(pixels))))
    return f


def extract_shape_features(mask):
    """形状特征（10 维）— 基于 regionprops 最大连通区域"""
    labeled = label(mask)
    regions = regionprops(labeled)
    f = {}
    if not regions:
        for k in ["area", "perimeter", "eccentricity", "major_axis",
                   "minor_axis", "solidity", "extent", "aspect_ratio",
                   "equiv_diameter", "convex_area"]:
            f[f"shape_{k}"] = 0.0
        return f

    region = max(regions, key=lambda r: r.area)
    f["shape_area"]           = float(region.area)
    f["shape_perimeter"]      = float(region.perimeter)
    f["shape_eccentricity"]   = float(region.eccentricity)
    f["shape_major_axis"]     = float(region.major_axis_length)
    f["shape_minor_axis"]     = float(region.minor_axis_length)
    f["shape_solidity"]       = float(region.solidity)
    f["shape_extent"]         = float(region.extent)
    minor = max(region.minor_axis_length, 1e-6)
    f["shape_aspect_ratio"]   = float(region.major_axis_length / minor)
    f["shape_equiv_diameter"] = float(region.equivalent_diameter_area)
    f["shape_convex_area"]    = float(region.convex_area)
    return f


def extract_glcm_features(image, mask):
    """GLCM 纹理特征（18 维）"""
    f = {}
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "asm"]:
            for d in [1, 2, 3]:
                f[f"glcm_{prop}_d{d}_mean"] = 0.0
        return f

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    crop_img  = image[rmin:rmax + 1, cmin:cmax + 1]
    crop_mask = mask[rmin:rmax + 1, cmin:cmax + 1]

    quantized = np.floor(crop_img.astype(np.float64) / 255.0 * 15).astype(np.uint8)
    quantized[crop_mask == 0] = 0

    distances = [1, 2, 3]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

    glcm = graycomatrix(quantized, distances=distances, angles=angles,
                         levels=16, symmetric=True, normed=True)

    for prop in props:
        prop_vals = graycoprops(glcm, prop)
        for d_idx, dist in enumerate(distances):
            f[f"glcm_{prop.lower()}_d{dist}_mean"] = float(np.mean(prop_vals[d_idx, :]))
    return f


def extract_all_features(image_path, mask_paths):
    """对单张图像提取全部影像组学特征（45 维）"""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None

    mask = np.zeros_like(image, dtype=np.uint8)
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if m.shape != image.shape:
            m = cv2.resize(m, (image.shape[1], image.shape[0]))
        mask = np.maximum(mask, (m > 0).astype(np.uint8))

    if mask.sum() == 0:
        return None

    roi_pixels = image[mask > 0].astype(np.float64)

    features = {}
    features.update(extract_first_order_features(roi_pixels))
    features.update(extract_shape_features(mask))
    features.update(extract_glcm_features(image, mask))
    return features


print("特征提取函数定义完成 (45 维)")

In [ ]:
# ====================
# 4. 批量提取所有样本特征
# ====================

feature_list = []
skipped = 0

for idx, row in df_data.iterrows():
    feats = extract_all_features(row["image_path"], row["mask_paths"])
    if feats is None:
        skipped += 1
        continue
    feats["label"] = row["label"]
    feats["label_name"] = row["label_name"]
    feature_list.append(feats)

    if (len(feature_list) % 100) == 0:
        print(f"  已处理 {len(feature_list)} / {len(df_data)} 个样本 ...")

df_features = pd.DataFrame(feature_list)
print(f"\n特征提取完成: {len(df_features)} 个样本, {df_features.shape[1] - 2} 维特征")
if skipped:
    print(f"跳过 {skipped} 个无效样本")

# 保存 CSV
output_csv = os.path.join(TASK_DIR, "busi_radiomics_features.csv")
df_features.to_csv(output_csv, index=False)
print(f"特征已保存至: {output_csv}")
df_features.head()

## 4. 特征相关性分析

在特征筛选之前，先检查 **45 维特征之间的相关性**。

**为什么要做这一步？**
- 高度相关的特征（|r| > 0.9）携带冗余信息，会增加模型方差
- 了解特征聚类结构，有助于理解后续 SelectKBest 的选择逻辑
- 在报告中展示热力图，体现对特征空间的深入理解

In [ ]:
# ====================
# 4.1 特征相关性热力图
# ====================

feature_cols = [c for c in df_features.columns if c not in ("label", "label_name")]
corr_matrix = df_features[feature_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask_upper, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            cbar_kws={"shrink": 0.6, "label": "Pearson r"},
            annot=False, ax=ax)
ax.set_title("Feature Correlation Matrix (45 features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 统计高相关特征对
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], r))

print(f"\n高相关特征对 (|r| > 0.9): {len(high_corr_pairs)} 对")
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2]))[:10]:
    print(f"  {f1:30s} ↔ {f2:30s}  r = {r:.3f}")

In [ ]:
# ====================
# 5. 数据预处理与特征筛选
# ====================

X = df_features[feature_cols].values
y = df_features["label"].values

print(f"特征矩阵: {X.shape}")
print(f"标签分布: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"类别比例: benign={np.sum(y==0)}, malignant={np.sum(y==1)} (不平衡比 ≈ {np.sum(y==0)/np.sum(y==1):.1f}:1)")

# 80/20 分层划分
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"\n训练集: {X_train.shape[0]}, 测试集: {X_test.shape[0]}")

# 标准化 (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 特征筛选 — SelectKBest (ANOVA F-test, k=20)
K_BEST = 20
selector = SelectKBest(score_func=f_classif, k=K_BEST)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel  = selector.transform(X_test_scaled)

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

print(f"\n选中 {len(selected_features)} 个特征 (ANOVA F-test):")
for i, feat in enumerate(selected_features, 1):
    score = selector.scores_[np.where(selected_mask)[0][i - 1]]
    print(f"  {i:2d}. {feat:30s}  F = {score:8.2f}")

## 6. 模型训练

训练 **3 种**分类模型，所有模型使用 `class_weight='balanced'` 处理类别不平衡：

| 模型 | 类型 | 关键超参数 | 不平衡处理 |
|------|------|-----------|-----------|
| **Logistic Regression** | 线性 | C=1.0, L2 正则 | `class_weight='balanced'` |
| **Random Forest** | 非线性 | n_estimators=200, max_depth=10 | `class_weight='balanced'` |
| **SVM (RBF)** | 非线性 | C=1.0, gamma=scale | `class_weight='balanced'` |

> **class_weight='balanced'** 的原理：自动调整每个类的权重，使少数类（malignant）
> 的每个样本获得更高权重 = 总样本数 / (类数 × 该类样本数)，
> 从而让模型不再偏向多数类。

In [ ]:
# ====================
# 6. 模型训练 (class_weight='balanced')
# ====================

lr_model = LogisticRegression(
    C=1.0, penalty="l2", solver="lbfgs",
    max_iter=2000, class_weight="balanced",
    random_state=RANDOM_STATE
)
lr_model.fit(X_train_sel, y_train)
print("LR 训练完成 (balanced)")

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train_sel, y_train)
print("RF 训练完成 (balanced)")

svm_model = SVC(
    C=1.0, kernel="rbf", gamma="scale",
    class_weight="balanced", probability=True,
    random_state=RANDOM_STATE
)
svm_model.fit(X_train_sel, y_train)
print("SVM 训练完成 (balanced)")

models = {
    "Logistic Regression": lr_model,
    "Random Forest":       rf_model,
    "SVM (RBF)":           svm_model,
}
print("\n所有模型训练完成!")

## 7. 五折交叉验证

**为什么需要交叉验证？**
- 单次 train/test 划分的结果受随机种子影响，可能偏高或偏低
- 5 折 CV 将训练集分成 5 份，每次用 4 份训练、1 份验证，重复 5 次
- CV-AUC 的均值±标准差更能反映模型的**真实泛化能力**

> 使用 `StratifiedKFold` 保持每折的类别比例一致。

In [ ]:
# ====================
# 7. 五折交叉验证评估
# ====================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for name, model in models.items():
    scores = cross_val_score(
        model, X_train_sel, y_train,
        cv=cv, scoring="roc_auc", n_jobs=-1
    )
    cv_results.append({
        "Model": name,
        "CV-AUC (mean)": scores.mean(),
        "CV-AUC (std)":  scores.std(),
        "CV scores":     ", ".join(f"{s:.4f}" for s in scores),
    })
    print(f"{name:25s}  CV-AUC = {scores.mean():.4f} ± {scores.std():.4f}")

df_cv = pd.DataFrame(cv_results)[["Model", "CV-AUC (mean)", "CV-AUC (std)"]]
print("\n" + df_cv.to_string(index=False))

## 8. 测试集评估

### 评估指标
| 指标 | 含义 | 医学意义 |
|------|------|---------|
| **AUC (95% CI)** | ROC 曲线下面积 + Bootstrap 置信区间 | 整体判别能力 |
| **Accuracy** | 整体分类正确率 | — |
| **Sensitivity** | TP/(TP+FN) | 恶性病灶被正确检出（**最重要**） |
| **Specificity** | TN/(TN+FP) | 良性病灶被正确排除 |

> **Bootstrap 95% CI**：对测试集进行 1000 次有放回重采样，计算每次的 AUC，
> 取 2.5% 和 97.5% 分位数作为置信区间。

In [ ]:
# ====================
# 8.1 性能指标汇总（含 Bootstrap AUC 95% CI）
# ====================

def bootstrap_auc(y_true, y_prob, n_boot=1000, seed=42):
    """Bootstrap 95% CI for AUC"""
    rng = np.random.RandomState(seed)
    aucs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)

results = []
for name, model in models.items():
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    auc_val = roc_auc_score(y_test, y_prob)
    ci_low, ci_high = bootstrap_auc(y_test, y_prob)

    results.append({
        "Model": name,
        "AUC":              f"{auc_val:.4f}",
        "AUC 95% CI":       f"[{ci_low:.4f}, {ci_high:.4f}]",
        "Accuracy":         f"{accuracy_score(y_test, y_pred):.4f}",
        "Sensitivity":      f"{tp/(tp+fn):.4f}",
        "Specificity":      f"{tn/(tn+fp):.4f}",
    })

df_results = pd.DataFrame(results)
print("=" * 80)
print("                     测试集性能对比 (with 95% CI)")
print("=" * 80)
print(df_results.to_string(index=False))
print("=" * 80)

best_idx = np.argmax([float(r["AUC"]) for r in results])
print(f"\n最佳模型 (by AUC): {results[best_idx]['Model']}")

In [ ]:
# ====================
# 8.2 ROC 曲线 & Precision-Recall 曲线
# ====================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for name, model in models.items():
    color = MODEL_COLORS[name]
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    # --- ROC ---
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax1.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc_val:.4f})")

    # --- PR ---
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax2.plot(recall, precision, color=color, lw=2, label=f"{name} (AP={ap:.4f})")

# ROC 参考线
ax1.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax1.set_xlabel("False Positive Rate (1 - Specificity)")
ax1.set_ylabel("True Positive Rate (Sensitivity)")
ax1.set_title("ROC Curves", fontweight="bold")
ax1.legend(loc="lower right")
ax1.set_xlim([-0.01, 1.01]); ax1.set_ylim([-0.01, 1.01])

# PR 参考线 (正类比例)
baseline = np.sum(y_test == 1) / len(y_test)
ax2.axhline(y=baseline, color="k", linestyle="--", lw=1, alpha=0.5, label=f"Baseline ({baseline:.2f})")
ax2.set_xlabel("Recall (Sensitivity)")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curves", fontweight="bold")
ax2.legend(loc="upper right")
ax2.set_xlim([-0.01, 1.01]); ax2.set_ylim([-0.01, 1.01])

plt.suptitle("Model Performance Comparison", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ====================
# 8.3 混淆矩阵
# ====================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = ["Benign", "Malignant"]

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_sel)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar=False, annot_kws={"size": 18})
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. 特征重要性分析

展示各模型中 **Top-10** 最具区分力的影像组学特征：
- **LR**：标准化后的回归系数绝对值 |β|
- **RF**：Gini 重要性 (feature_importances_)

> 两个模型共同认为重要的特征，可信度更高。

In [ ]:
# ====================
# 9. Top-10 特征重要性图（LR + RF 对比）
# ====================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# LR
lr_coefs = np.abs(lr_model.coef_[0])
lr_imp = pd.Series(lr_coefs, index=selected_features).sort_values(ascending=False).head(10)
axes[0].barh(range(len(lr_imp)), lr_imp.values, color=MODEL_COLORS["Logistic Regression"], edgecolor="white")
axes[0].set_yticks(range(len(lr_imp)))
axes[0].set_yticklabels(lr_imp.index, fontsize=10)
axes[0].invert_yaxis()
axes[0].set_xlabel("|Coefficient| (standardized)")
axes[0].set_title("Logistic Regression — Top 10", fontweight="bold")

# RF
rf_imp = pd.Series(rf_model.feature_importances_, index=selected_features)
rf_imp = rf_imp.sort_values(ascending=False).head(10)
axes[1].barh(range(len(rf_imp)), rf_imp.values, color=MODEL_COLORS["Random Forest"], edgecolor="white")
axes[1].set_yticks(range(len(rf_imp)))
axes[1].set_yticklabels(rf_imp.index, fontsize=10)
axes[1].invert_yaxis()
axes[1].set_xlabel("Gini Importance")
axes[1].set_title("Random Forest — Top 10", fontweight="bold")

plt.suptitle("Feature Importance Comparison", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 找出两个模型共同的重要特征
lr_top = set(lr_imp.index[:10])
rf_top = set(rf_imp.index[:10])
common = lr_top & rf_top
print(f"\n两模型共同认为重要的特征 ({len(common)} 个):")
for feat in common:
    print(f"  - {feat}")

## 10. 学习曲线分析

学习曲线展示：随着训练样本量增加，训练集和验证集的 AUC 如何变化。

**诊断逻辑**：
- **训练集高、验证集低** → 过拟合（需要更多数据或正则化）
- **两者都低** → 欠拟合（需要更复杂模型或更多特征）
- **两者趋近且较高** → 理想状态

In [ ]:
# ====================
# 10. 学习曲线（以 RF 为例）
# ====================

train_sizes = np.linspace(0.1, 1.0, 10)
sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    X_train_sel, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc", train_sizes=train_sizes, n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 6))

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

ax.plot(sizes, train_mean, "o-", color=MODEL_COLORS["Random Forest"], lw=2, label="Training AUC")
ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color=MODEL_COLORS["Random Forest"])

ax.plot(sizes, val_mean, "s--", color="#E91E63", lw=2, label="Validation AUC")
ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#E91E63")

ax.set_xlabel("Training Set Size")
ax.set_ylabel("AUC")
ax.set_title("Learning Curve — Random Forest", fontweight="bold")
ax.legend(loc="lower right")
ax.set_ylim([0.7, 1.01])
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"最终训练 AUC: {train_mean[-1]:.4f}")
print(f"最终验证 AUC: {val_mean[-1]:.4f}")
print(f"过拟合间隔:  {gap:.4f}")
if gap > 0.1:
    print("→ 存在过拟合迹象，可考虑增加正则化或减少特征数")
elif val_mean[-1] < 0.8:
    print("→ 存在欠拟合迹象，可考虑增加模型复杂度")
else:
    print("→ 模型拟合状态良好")

## 11. 可解释性分析 — SHAP

**SHAP** (SHapley Additive exPlanations) 基于博弈论 Shapley 值，
为每个预测提供特征级别的贡献分解。

> 需要安装：`pip install shap`

In [ ]:
# ====================
# 11. SHAP 可解释性分析
# ====================

try:
    import shap

    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_sel)

    # 二分类: shap_values 可能是 list[2]，取 class=1
    if isinstance(shap_values, list):
        sv_plot = shap_values[1]
    else:
        sv_plot = shap_values

    # --- Summary Plot ---
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv_plot, X_test_sel,
                      feature_names=selected_features, show=False)
    plt.title("SHAP Summary — Random Forest", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # --- 个体预测解释 ---
    malignant_idx = np.where(y_test == 1)[0][0]
    plt.figure(figsize=(10, 4))
    if isinstance(shap_values, list):
        sv_ind = shap_values[1][malignant_idx]
    else:
        sv_ind = shap_values[malignant_idx]

    expected = explainer.expected_value
    if isinstance(expected, (list, np.ndarray)):
        expected = expected[1]

    shap.force_plot(expected, sv_ind, X_test_sel[malignant_idx],
                    feature_names=selected_features,
                    matplotlib=True, show=False)
    plt.title(f"Individual Explanation (Sample #{malignant_idx}, True=Malignant)")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("shap 未安装。请运行: pip install shap")
except Exception as e:
    print(f"SHAP 分析出错: {e}")

## 12. 超参数优化 (GridSearchCV)

使用 **5 折分层交叉验证** 对 Random Forest 进行网格搜索：
- 搜索 36 种参数组合 × 5 折 = 180 次训练
- 优化目标：AUC

In [ ]:
# ====================
# 12. GridSearchCV 超参数优化
# ====================

param_grid = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_sel, y_train)

print(f"\n最佳参数: {rf_grid.best_params_}")
print(f"最佳 CV AUC: {rf_grid.best_score_:.4f}")

# 测试集评估
best_rf = rf_grid.best_estimator_
y_pred_best = best_rf.predict(X_test_sel)
y_prob_best = best_rf.predict_proba(X_test_sel)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_best).ravel()

ci_low, ci_high = bootstrap_auc(y_test, y_prob_best)

print(f"\n优化后 RF 测试集性能:")
print(f"  AUC         = {roc_auc_score(y_test, y_prob_best):.4f}  (95% CI: [{ci_low:.4f}, {ci_high:.4f}])")
print(f"  Accuracy    = {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  Sensitivity = {tp/(tp+fn):.4f}")
print(f"  Specificity = {tn/(tn+fp):.4f}")

# 对比优化前后
print(f"\n优化前 RF AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test_sel)[:,1]):.4f}")
print(f"优化后 RF AUC: {roc_auc_score(y_test, y_prob_best):.4f}")

## 13. 总结与讨论

### 实验结果回顾
（运行 notebook 后，在此填写你的实验结果）

| 模型 | CV-AUC | 测试 AUC (95% CI) | Accuracy | Sensitivity | Specificity |
|------|--------|-------------------|----------|-------------|-------------|
| LR   |  —     |  —  (—)          |   —      |     —       |     —       |
| RF   |  —     |  —  (—)          |   —      |     —       |     —       |
| SVM  |  —     |  —  (—)          |   —      |     —       |     —       |

### 讨论要点

**1. 哪种模型表现最好？**
- 对比 CV-AUC 和测试集 AUC 的稳定性
- class_weight='balanced' 对 Sensitivity 的提升效果
- 线性 vs 非线性模型的 trade-off

**2. 哪些影像组学特征较重要？**
- 两模型共同选出的重要特征（见特征重要性分析）
- 一阶统计：entropy / kurtosis → 肿瘤内部异质性
- GLCM：contrast / correlation → 纹理粗糙度与方向性
- 形状：eccentricity / solidity → 边缘不规则性

**3. 类别不平衡的影响**
- 使用 class_weight='balanced' 后，Sensitivity 是否提升？
- PR 曲线相比 ROC 更能反映少数类的分类效果

**4. 模型局限性**
- BUSI 类别不平衡 (2:1)，虽用 balanced 处理但仍有偏差
- 2D 超声图像，未利用 3D / 多模态信息
- 缺乏外部验证集
- 手动 mask 标注质量影响特征提取
- 特征提取参数（GLCM 距离、量化级数）未优化

### 本实验的优化亮点
- ✅ **类别不平衡处理**：class_weight='balanced'
- ✅ **特征相关性分析**：45维热力图，识别冗余特征
- ✅ **5折交叉验证**：CV-AUC 均值±标准差
- ✅ **Bootstrap AUC 95% CI**：统计严谨性
- ✅ **PR 曲线**：不平衡数据的补充评估
- ✅ **学习曲线**：诊断过拟合/欠拟合
- ✅ **SHAP 可解释性**：全局 + 个体层面
- ✅ **GridSearchCV 超参数优化**：系统性搜索